> [!IMPORTANT]
> **Disclaimer**: The content and views presented during this session are the author's own and not of any organizations they are associated with or employed at. The code shown in this repository is for illustration and educational purposes only. It is not production-grade; error handling, security, and scalability are not fully addressed.

# Lab 3: Agent Metrics, Evaluations, Alerting, and Profiling
**Faculty Development Programme on Observability for AI Agents**

> [!NOTE]
> **Curriculum Cross-Reference**: This laboratory exercise implements the practical aspects of the **Metrics, Alerting, and Profiling** slides detailed in **Section 4 of [00_curriculum_and_agenda.ipynb](00_curriculum_and_agenda.ipynb)**.

### Overview
Structured logging and distributed tracing facilitate the isolation and debugging of individual transaction failures. However, maintaining system reliability across large-scale deployments requires:
1. **Aggregated Metrics**: Aggregated Metrics (consolidated statistics representing system performance over a timeframe). Production architectures monitor client-side user experience (UX) metrics alongside server-side operational metrics.
2. **Execution Profiling**: Execution Profiling (runtime analysis to isolate CPU bottlenecks and resource hotspots, e.g., synchronous blocking in tool executions).
3. **Automated Evaluations**: Heuristics to measure response accuracy, bias, and output groundedness (verifying if a generated response is strictly supported by retrieved source context documents without hallucinations).
4. **Alerting Rules**: Diagnostic triggers that fire when system metrics exceed defined operational thresholds.

---

### 📊 Key Production Metrics

#### 📱 Client-Side UX Performance
* **Time to First Token (TTFT)**: Time to First Token (TTFT - the duration elapsed until the initial token streams to the client, critical for user responsiveness).
* **Inter-Token Latency**: Inter-Token Latency (the average time elapsed between subsequent generated tokens, determining output readability smoothness).
* **End-to-End Latency**: Total duration of the round-trip API request lifecycle.
* **Network Overhead**: Delay introduced during transport layer transmission relative to model processing times.

#### 🖥️ Server-Side (Agentic) Performance
* **Tool Latency Breakdown**: Latency distributions mapped per tool class (e.g., search queries vs. local database lookups).
* **Reasoning Loop Step Count**: Reasoning Loop Step Count (the total number of iterations or thoughts the agent executes before completing a task).
* **Token Throughput**: Tokens processed and generated per unit time.
* **KV Cache Utilization and Hit Rate**: Key-Value Cache Utilization and Hit Rate (tracking GPU block memory allocations storing computed context tokens. Cache exhaustion degrades latency by triggering context evictions or queue bottlenecks, whereas high hit rates reduce TTFT).
* **Semantic Cache Savings**: Savings Telemetry (tracking cumulative token expenditures, processing latency, and carbon emissions saved by cache hits that avoided model generation queries).
* **FinOps (Cost Metrics)**: Cumulative cost expenditures tracked per session or per agent version.
* **GreenOps (Environmental Metrics)**: Cumulative energy consumed (kWh) and carbon footprint emissions ($CO_2e$).

---

### 🔍 Client vs. Server Profiling Boundaries
* **Client-Side Profiling**: Targets network transport handshake times, local streaming buffers, and payload parsing delays.
* **Server-Side Profiling**: Targets synchronous blocking calls within tool blocks, unoptimized library routines, and memory leaks in chat history buffers.

---

### Learning Objectives:
1. Initialize the OpenTelemetry Metrics SDK with in-memory readers.
2. Register and record core operational telemetry counters and histograms.
3. Profile execution code paths using Python's native `cProfile` and `pstats` modules to isolate slow functions.
4. Implement a local automated evaluation script to verify response correctness.
5. Code threshold-based alerting checks to detect latency and expenditure anomalies.

## 1. Setting up OpenTelemetry Metrics SDK

OpenTelemetry enables the declaration and recording of metrics such as:
* **Counters**: Values that only increase (e.g., number of agent runs, error counts).
* **Histograms**: Distributions of values (e.g., LLM response latencies, token counts per request).

### 📊 In-Memory Metric Readers vs. Production Alternatives
To aggregate and export metric datasets, OpenTelemetry utilizes **Metric Readers**. In this laboratory exercise, an **`InMemoryMetricReader`** is initialized:
* **Why In-Memory?**: It keeps collected metrics in local thread-safe memory. This enables the notebook to programmatically fetch, inspect, and print metric values directly within subsequent execution cells (using `reader.get_metrics_data()`) without requiring any running telemetry backend.

In production architectures, developers replace the in-memory reader with industry alternatives:
1. **Prometheus Metric Reader (`PrometheusMetricReader` - Pull Model)**:
   - *How it works*: Exposes an HTTP endpoint (commonly `/metrics`) on the application server. Prometheus scrapers connect periodically to pull the latest metric values.
   - *Use case*: The standard pattern for Kubernetes and persistent virtual machine deployments.
2. **OTLP Metric Exporter (`OTLPMetricExporter` - Push Model)**:
   - *How it works*: Periodically pushes accumulated metrics (e.g., every 60 seconds) over gRPC or HTTP/JSON protocols to an OpenTelemetry Collector or cloud telemetry backend (such as Google Cloud Monitoring or Grafana Cloud).
   - *Use case*: Recommended for serverless Platform as a Service (PaaS) and Function as a Service (FaaS) topologies (e.g., AWS Lambda, GCP Cloud Run) where hosts are ephemeral and cannot be scraped.
3. **Console Metric Exporter (`ConsoleMetricExporter`)**:
   - *How it works*: Periodically serializes and prints active metric records directly to standard output (`stdout`).
   - *Use case*: Used for local testing and process-level debugging.

In [ ]:
import sys
import os
import time
import cProfile
import pstats
import io
from typing import Dict, Any, Optional
sys.path.append(os.path.abspath('..'))

from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from src.mock_llm import MockLLMClient, MockLLMResponse

# Instantiates an InMemoryMetricReader to capture metrics in RAM for direct polling/reading in the notebook
reader = InMemoryMetricReader()
# Creates the central MeterProvider configured with our metric reader
provider = MeterProvider(metric_readers=[reader])
# Sets this provider instance as the global metrics manager for the application
metrics.set_meter_provider(provider)

# Acquires a named meter instance to define and record metric objects
meter = metrics.get_meter("agent_metrics")

# Registers a Counter metric to track the total cumulative count of agent runs completed
run_counter = meter.create_counter(
    name="agent_runs_total",
    description="Total number of agent runs completed",
    unit="1"
)

# Registers a Counter metric to track the total cumulative count of failed agent executions
error_counter = meter.create_counter(
    name="agent_errors_total",
    description="Total number of agent runs that resulted in an error",
    unit="1"
)

# Registers a Histogram metric to measure distribution parameters (average, min, max, bounds) of LLM generation latencies
latency_histogram = meter.create_histogram(
    name="llm_latency_seconds",
    description="Distribution of LLM response latencies",
    unit="s"
)

# Registers a Counter metric to track cumulative financial API expenditure costs of agent generation loops
cost_counter = meter.create_counter(
    name="agent_cost",
    description="Accumulated cost",
    unit="cost"
)

print("Metrics SDK initialized. Counters and Histograms registered.")

> [!IMPORTANT]
> **Observability Insight: Why Metrics are Not Visible in Arize Phoenix**
> * **The Observation**: If you check the Arize Phoenix UI dashboard (at `http://localhost:6006`) after executing the traffic simulation above, you will **not** find any metrics graphs or counters.
> * **The Explanation**: In this lab, the OpenTelemetry Metrics SDK is configured to use the **`InMemoryMetricReader`**. This reader gathers metrics strictly within the notebook process's thread memory. Because no exporter (such as OTLP Metrics Exporter) is configured to push data over the network, these metrics remain local. 
> * **How We Read Them**: The metrics are programmatically pulled from the local memory buffer and printed to the cell output using Python code:
>   `reader.get_metrics_data()`
> * **Production Setup**: In production, metrics are not stored in-memory. The Metrics SDK is configured with a `PrometheusMetricReader` (pull model) or an `OTLPMetricExporter` (push model) to route metric streams to database engines (such as Prometheus or Google Cloud Monitoring) for visualization on Grafana dashboards.

## 2. Instrumenting the Agent Loop to Record Metrics

A wrapper function is implemented to execute the mock LLM, record metrics (such as response latency and cost), and increment the execution counter. A deterministic assertion check is also performed to evaluate the accuracy of the model's output.

### 🤖 Telemetry Ingest Provider: MockLLMClient
To gather aggregate latency, token volumes, and financial transaction metrics, this lab uses the local **`MockLLMClient`** (see **[Lab 1](01_structured_logging_cost_carbon.ipynb)** for full implementation context). The client emulates variable request latency (recorded inside OTel histograms) and increments cost metrics (accumulated inside OTel cost counters) without generating live API charges.

In [2]:
# The MockLLMClient is initialized
llm_client: MockLLMClient = MockLLMClient()

def run_agent_with_metrics(query: str, expected_answer: Optional[str] = None) -> Dict[str, Any]:
    """
    Executes a model query transaction, recording structured aggregate metrics
    (counters, latency histograms, and costs) to the OpenTelemetry registry.
    Additionally evaluates response accuracy if expected answer labels are provided.

    Args:
        query (str): The user query input string.
        expected_answer (Optional[str]): Ground truth answer string for deterministic evaluation.

    Returns:
        Dict[str, Any]: Execution details including latency, cost, correctness status, and output text.
    """
    # The global execution count metric is incremented
    run_counter.add(1)

    start_time: float = time.time()
    try:
        prompt: str = f"System: Answer the user question.\nUser: {query}"

        llm_start: float = time.time()
        response: MockLLMResponse = llm_client.generate(prompt)
        llm_latency: float = time.time() - llm_start

        # Latency distribution metrics are captured
        latency_histogram.record(llm_latency)

        # Financial consumption costs are accumulated
        cost_counter.add(response.cost)

        total_latency: float = time.time() - start_time

        # An automated exact-match correctness evaluation is executed
        correct: bool = False
        if expected_answer:
            correct = expected_answer.lower() in response.text.lower()

        return {
            "text": response.text,
            "latency": total_latency,
            "cost": response.cost,
            "evaluation_correct": correct,
            "status": "success"
        }

    except Exception as e:
        # The runtime execution error count metric is incremented
        error_counter.add(1)
        return {
            "text": "",
            "latency": time.time() - start_time,
            "cost": 0.0,
            "evaluation_correct": False,
            "status": f"error: {str(e)}"
        }

## 3. Simulating Traffic and Reading the Metrics

A batch of queries is simulated. Following execution, the OpenTelemetry in-memory metric reader is inspected to retrieve the recorded values, displaying the average latency, total error count, and cumulative cost.

In [6]:
# Run multiple queries
queries_to_run = [
    {"q": "What is the capital of France?", "ans": "Paris"},
    {"q": "Explain quantum computing", "ans": "quantum states"},
    {"q": "What is the capital of France?", "ans": "Paris"},
    {"q": "Trigger an error (simulate unknown behavior)", "ans": "None"} # this query will hit our default response
]

print("Simulating traffic...")
results = []
for item in queries_to_run:
    res = run_agent_with_metrics(item["q"], item["ans"])
    results.append(res)
    print(f"Query: {item['q'][:30]}... | Status: {res['status']} | Cost: ${res['cost']:.6f} | Eval Correct: {res['evaluation_correct']}")

print("\n--- Pulling Metrics from OpenTelemetry Reader ---")
# Retrieve the metrics from the in-memory reader
metric_data = reader.get_metrics_data()

# Parse metrics output to display nicely
for resource_metric in metric_data.resource_metrics:
    for scope_metric in resource_metric.scope_metrics:
        for metric in scope_metric.metrics:
            print(f"\nMetric Name: {metric.name}")
            print(f"Description: {metric.description}")
            for point in metric.data.data_points:
                # Print attributes, if any
                attributes = point.attributes
                # Print value based on metric data type (sum vs histogram)
                if hasattr(point, "value"):
                    print(f"Value: {point.value}")
                elif hasattr(point, "sum"):
                    # For histograms
                    print(f"Sum: {point.sum:.4f}")
                    print(f"Count: {point.count}")

Simulating traffic...
Query: What is the capital of France?... | Status: success | Cost: $0.000675 | Eval Correct: True
Query: Explain quantum computing... | Status: success | Cost: $0.002940 | Eval Correct: True
Query: What is the capital of France?... | Status: success | Cost: $0.000675 | Eval Correct: True
Query: Trigger an error (simulate unk... | Status: success | Cost: $0.002175 | Eval Correct: False

--- Pulling Metrics from OpenTelemetry Reader ---

Metric Name: agent_runs_total
Description: Total number of agent runs completed
Value: 12

Metric Name: llm_latency_seconds
Description: Distribution of LLM response latencies
Sum: 4.9429
Count: 12

Metric Name: agent_cost
Description: Accumulated cost
Value: 0.019395000000000003


## 4. Server-Side Profiling, Thread Dumps, and Memory Audits

While aggregated metrics define system performance over a timeframe, **profiling** serves as the diagnostic mechanism to isolate *why* specific code paths are slow. If agent execution latency spikes, profiling determines if the bottleneck resides within the model generation step or a poorly implemented local execution tool.

Production systems leverage multiple profiling paradigms:
1. **CPU Execution Profiling**: Records function call frequencies and execution durations. This laboratory exercise utilizes Python's built-in `cProfile` and `pstats` modules to identify CPU cycles hotspots.
2. **Thread Dumps**: Captures active stack trace states of all executing threads (e.g., via Python's `faulthandler` or JVM `jstack`). This is essential to diagnose socket read lockups (e.g., waiting indefinitely on a remote API response) or thread deadlocks in parallel tool executions.
3. **Heap Dumps and Memory Audits**: Measures memory allocations on the heap (e.g., via Python's `tracemalloc`). This isolates memory leaks resulting from dynamic chat history arrays or unbounded context windows.

In [7]:
def slow_agent_task() -> None:
    """
    Simulates a combined agent task containing both a mock language model call
    and a slow database lookup or execution bottleneck.
    """
    print("Executing agent loop...")
    # Inference generation step simulation
    llm_client.generate("What is the population of France?")

    # Inefficient external database query tool simulation
    time.sleep(0.8)
    print("Agent loop finished.")

# The CPU runtime execution profiler is configured and enabled
pr = cProfile.Profile()
pr.enable()

# The task instance under observation is executed
slow_agent_task()

pr.disable()

# Execution metrics are collected and formatted
s = io.StringIO()
sortby = pstats.SortKey.TIME
ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
ps.print_stats(10) # The top 10 most expensive functions are extracted

print("\n--- TOP 10 CALLS BY TOTAL TIME ---")
print(s.getvalue()[:800])

Executing agent loop...
Agent loop finished.

--- TOP 10 CALLS BY TOTAL TIME ---
         1231 function calls (1221 primitive calls) in 1.124 seconds

   Ordered by: internal time
   List reduced from 211 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        6    0.685    0.114    0.685    0.114 {method 'poll' of 'select.epoll' objects}
        2    0.426    0.213    0.426    0.213 {built-in method time.sleep}
        2    0.009    0.004    0.009    0.004 {method '__exit__' of 'sqlite3.Connection' objects}
        1    0.001    0.001    0.010    0.010 /usr/local/google/home/anmolsachdeva/mnit-aiml-fdp-may-2026/venv/lib/python3.13/site-packages/IPython/core/history.py:100(only_when_enabled)
       24    0.000    0.000    0.000    0.000 /usr/local/google/home/anmolsachdeva/mnit-aiml-fdp-may-2026/venv/lib/python3.13


### 💡 How to Interpret Python cProfile Output
When analyzing the profiler statistics printed above, look at the columns to understand where execution time is concentrated:

1. **Understanding the Columns**:
   * **`ncalls`**: The number of times the function was called.
   * **`tottime`**: The total time spent inside the function itself, **excluding** time spent in other sub-functions it called. (This is the most critical column for finding local code bottlenecks).
   * **`cumtime`**: The cumulative time spent in the function **including** all sub-functions it invoked. (Useful to see the total path latency).
   * **`percall`**: Average duration per invocation (calculated as `tottime / ncalls` or `cumtime / ncalls`).
   * **`filename:lineno(function)`**: Indicates the script path, line number, and function name.

2. **Understanding the Call Summary (e.g., "1231 function calls (1221 primitive calls)")**:
   * **Why are there so many calls?**: Even though only a single high-level function (`slow_agent_task()`) was called directly, Python executes numerous underlying functions under the hood. For instance, compiling regular expressions, formatting string prompts, checking dictionaries, and executing sleep cycles all invoke secondary helper routines. Additionally, because this executes inside a Jupyter Notebook context, the notebook's background helper threads (e.g., cell history logging and printing managers) also add to the recorded total.
   * **What is a "Primitive Call"?**: A primitive call is a function call that is **not recursive** (i.e., the function did not invoke itself). The difference between the total calls and primitive calls (e.g., $1231 - 1221 = 10$) represents recursive stack calls executed during string matching or serialization.

3. **Analyzing the Results**:
   * The top bottleneck identified is **`{built-in method time.sleep}`** with a `tottime` of **~0.685 seconds** (or similar depending on local run latency).
   * *The Cause*: This represents the simulated delay inside the local mock model client (`MockLLMClient.generate`) combined with the `time.sleep(0.8)` statement mimicking a slow tool database query.
   * *The Remediation*: In a production profile, finding high `tottime` on socket reads (`select.epoll` or `socket.recv`) indicates that the application is blocking on external HTTP model endpoints or downstream datastores, signaling that async calling, database indexing, or caching is required.

## 5. Implementing Basic Alerting Rules

In production, alerting engines (such as Prometheus Alertmanager or Grafana Alerts) are configured to evaluate metrics periodically.
A simulated alerting function is executed following request processing to verify if metric values exceed defined thresholds:
* Latency threshold: 0.5 seconds
* Cost threshold: 0.0030 (generic units)

In [8]:
# Define alert thresholds for latency and API financial expenditure
LATENCY_ALERT_THRESHOLD_SECONDS: float = 0.5
COST_ALERT_THRESHOLD: float = 0.0030

def check_alerting_rules(run_result: Dict[str, Any]) -> None:
    """
    Evaluates individual execution stats against preset alert thresholds.
    Triggers diagnostic notifications when thresholds are crossed.

    Args:
        run_result (Dict[str, Any]): Dictionary containing latency, cost, and status.
    """
    alerts = []
    if run_result["latency"] > LATENCY_ALERT_THRESHOLD_SECONDS:
        alerts.append(
            f"⚠️ ALERT: High Latency detected! Latency was {run_result['latency']:.2f}s "
            f"(Threshold: {LATENCY_ALERT_THRESHOLD_SECONDS}s)"
        )
    if run_result["cost"] > COST_ALERT_THRESHOLD:
        alerts.append(
            f"🚨 ALERT: Cost Limit Exceeded! Cost was {run_result['cost']:.6f} "
            f"(Threshold: {COST_ALERT_THRESHOLD:.6f})"
        )

    if alerts:
        print("\n".join(alerts))
    else:
        print("✅ Run parameters within normal operating limits.")

# Test alerting rules on previous diagnostic runs
print("Checking previous run results against alert thresholds:\n")
for i, res in enumerate(results):
    print(f"--- Query {i+1} Alerting Check ---")
    check_alerting_rules(res)

Checking previous run results against alert thresholds:

--- Query 1 Alerting Check ---
✅ Run parameters within normal operating limits.
--- Query 2 Alerting Check ---
✅ Run parameters within normal operating limits.
--- Query 3 Alerting Check ---
✅ Run parameters within normal operating limits.
--- Query 4 Alerting Check ---
✅ Run parameters within normal operating limits.


> [!WARNING]
> **Production Security Note: Sandbox Isolation and Latent Threat Remediation Across Topologies**
> * **The Threat**: AI Agents with active execution capabilities (such as writing files and running code) can be exploited via code-execution injections or jailbreaks. If the agent runs untrusted code inside a poorly isolated environment, it can cause Denial of Service (DoS) resource exhaustion or leak host system secrets.
> * **Leading Remediation Practices (By Deployment Topology)**:
>   To enforce secure boundary runtime isolation, security teams deploy controls tailored to the specific platform:
>   1. **Local and Bare-Metal**: Wrap untrusted executions inside chroot jails, systemd-nspawn containers, or restricted user profiles with strict disk write quotas.
>   2. **Virtual Machines (VMs) and Kubernetes (K8s)**: Deploy code execution tools inside microVM instances (e.g., AWS Firecracker) or kernel-space sandboxed runtimes (e.g., gVisor, WebAssembly (WASM) runtimes) configured with read-only root filesystems and tight resource limits.
>   3. **PaaS, Serverless, and FaaS**: Rely on cloud provider microVM hypervisors (e.g., AWS Lambda Firecracker microVMs) but enforce strict function execution timeout caps (e.g., maximum 5 seconds execution life), memory limitations, and read-only environment variables.
>   4. **Unified Application Controls (All Platforms)**: Regardless of infrastructure, run inline safety guardrails (e.g., Llama Guard or NeMo Guardrails) at pre-prompt and post-generation execution gates to check for toxic inputs, PII leaks, and injection scripts.

## 6. Production Guidelines and Leading Practices

Deploying metrics, alerting, and profiling in high-scale agent workloads requires balancing observability depth with system overhead:

### 📊 Metric Collection and Alerting
* **Asynchronous Scrape Endpoints**: Avoid pushing metrics synchronously to central servers. Instead, configure Prometheus to scrape an HTTP `/metrics` endpoint exposed by the agent container asynchronously.
* **Alert Routing and Alertmanager**: Route PromQL alerts to a dedicated alerting manager (e.g., Prometheus Alertmanager, PagerDuty, or Opsgenie) to group duplicate notifications, apply silences, and route high-severity warnings to on-call schedules.
* **PromQL Alert Rules**:
  * *Error Spikes*: `rate(agent_errors_total[5m]) > 0.05` (alerts if error rate exceeds 5% of runs).
  * *Latency Threshold*: `histogram_quantile(0.95, sum(rate(llm_latency_seconds_bucket[5m])) by (le)) > 2` (alerts if the 95th percentile latency of LLM calls exceeds 2 seconds).

### 🔍 Production Profiling
* **Asynchronous Continuous Profiling**: Continuous usage of synchronous utilities like `cProfile` introduces severe execution overhead (up to 20% latency inflation) and is unsuitable for production. Recommend using **Continuous Profiling Agents** (e.g., Pyroscope or Google Cloud Profiler) which sample thread call stacks asynchronously at regular intervals, maintaining CPU overhead below 1%.
* **Memory Heap Auditing**: Integrate heap profiling tools during pre-production stress tests to identify context retention memory leaks, rather than continuously capturing memory snapshots in live customer pathways.

### Summary of Lab 3
1. **Aggregated Metrics**: Counters and histograms provide an operational dashboard view (average latency, total costs, and error rates).
2. **Profiling**: Using tools like `cProfile` helps developers surgically dissect latency and locate the specific tool or database query causing performance bottlenecks.
3. **Evaluations**: Validating answers programmatically using assertion/eval modules measures agent accuracy and regression.
4. **Alerting**: Alerting rules flag anomalous behavior (e.g., infinite reasoning loops or spiking costs) automatically.

This completes the three hands-on laboratory modules of the Observability for AI Agents curriculum.